# NBA Model v2 on Colab

End-to-end, evidence-first Model v2 workflow:

1. Clone or sync the repository, then preflight that the ref actually contains Model v2
2. Mount Google Drive for persistence
3. Install the lightweight v2/data-capture dependency set
4. Configure Drive paths and v2 run parameters
5. Update source data and create an immutable snapshot (or reuse one)
6. Capture official schedule/roster tables and rebuild a snapshot that carries them
7. Canonicalize and check the snapshot
8. Train and validate an immutable candidate bundle
9. Simulate scheduled games (official, shadow, or explicitly degraded)
10. Locate the immutable forecast files
11. Answer probability queries through `query_prob.py`

**Which ref contains Model v2?** Model v2 is shipped on the pushed `V2-WIP` branch,
so the notebook defaults `REPO_REF` to `V2-WIP` and then preflights the clone for the
required v2 files and CLI modules. If you point `REPO_REF` at another ref and those
files are absent, the preflight stops with a message telling you to set `REPO_REF` back
to the pushed `V2-WIP` ref that carries Model v2. This notebook does not commit or push
anything.

**Honest limits of this stack**

- The supported runtime is **Model v2 only**. Legacy CatBoost/Transformer presets, `S/M/L/XL` tiers, `check_contracts.py`, `ProjectionLoader`/`ProbabilityCalculator`/`InteractiveCLI`, and query `--data-dir` paths are retired and intentionally absent here.
- The lightweight rolling baseline is **not** promotion eligible: it is built from appearance-only history, so it has no official point-in-time roster/status data and no multi-fold replay evidence. Its candidate JSON reports `promoted: false`, and that gate is never weakened below.
- The supported v2 baseline does **not** use a GPU. Colab GPU acceleration buys nothing for this preset.
- Degraded runs use appearance-derived rosters, are labeled as degraded, and never publish to the official ledger.

**Snapshots.** `update_data.py` refreshes the working CSVs and, with `--snapshot`, freezes
them into a checksummed source snapshot. Snapshots and canonical tables are immutable and
content-addressed, so a snapshot is created once and reused. Set `SNAPSHOT_ID` plus
`REUSE_EXISTING_SNAPSHOT = True` in the configuration cell to train against a snapshot you
already have.

**The schedule gap.** `simulate_season.py` forecasts *scheduled* games and reads that
schedule out of the snapshot (`schedule.csv` / `nba_schedule.csv`). A snapshot created by
`update_data.py` contains only the working files present in your data directory, so a plain
refresh usually produces a snapshot with **no schedule**, and simulation then fails with
"No scheduled games found" or a missing-schedule contract error. Official capture is a
two-step, receipt-time-honest process:

1. `capture_official.py --native schedule --season YYYY-YY` and, when you need official
   roster membership, `capture_official.py --native rosters --season YYYY-YY` each archive
   one table into an immutable capture snapshot under `data/official_captures/`.
2. `rebuild_snapshot.py --base-snapshot-id <snapshot> --capture <schedule dir> --capture
   <roster dir>` writes **one new** snapshot carrying both tables, leaving the base snapshot
   intact.

The capture cell captures a **fresh** schedule by default and only captures rosters when
`MODE` is `official` or `shadow`. A full roster capture costs one rate-limited request per
team (30 teams), so the default `degraded` mode does not spend them.

**Strict mode is a same-day contract.** The official roster reader rejects a capture whose
Eastern receipt date differs from the forecast cutoff's Eastern date
("Official roster requires a current-day capture"), so a single roster capture only
certifies the requests whose cutoff falls on that same local date. A `week` or `season`
strict run spans several local dates and therefore needs a separate point-in-time snapshot
and roster capture per date; one notebook capture cannot cover it. Membership itself is
open-ended from the capture date, but that freshness rule still binds. `degraded` remains
the practical default here.

**Horizon cutoff rule (read before capturing).** A snapshot is usable for a horizon only
when **both** its manifest `created_at` **and** every file's receipt time are at or before
that horizon's forecast cutoff. `rebuild_snapshot.py` stamps a new `created_at` at rebuild
time, so the rebuild itself must happen before the cutoff. Cutoffs are Eastern:

| Horizon | Cutoff |
| --- | --- |
| `previous_night` | 23:59 ET the day before the game date |
| `morning` | 09:00 ET on the game date |
| `pregame_90m` | 90 minutes before tipoff |
| `pregame_30m` | 30 minutes before tipoff |

For a same-day capture, that means the schedule/roster capture **and** the rebuild must
finish before 09:00 ET to run the `morning` horizon on that day's games. A capture taken
after the cutoff cannot be used for that horizon at all, no matter how recent it is. The
simulation cell preflights this and refuses to launch a run that cannot satisfy the cutoff.

**Simulation modes.** `simulate_season.py` classifies every run and reports it in its JSON:

| Mode | Flags | Ledger |
| --- | --- | --- |
| `official` | none | publishes, requires point-in-time roster membership |
| `shadow` | `--candidate <dir>` | candidate evidence only, never publishes |
| `degraded` | `--candidate <dir> --allow-degraded` | appearance-derived rosters, never publishes |

With no champion configured in `models/champion.json` an official run fails, and degraded
runs need **both** flags: `--candidate` supplies the sealed bundle and `--allow-degraded`
unlocks appearance-derived rosters. The `published_to_official_ledger` field echoed by the
CLI is advisory: it reflects the mode the run was classified as, not independent proof that
a ledger write completed.

**Querying.** `query_prob.py` reads an immutable forecast parquet, or the official ledger,
with `--forecast-file`. `--players-file` is passed explicitly so name lookup never depends
on the working directory.

In [ ]:
# @title Clone or sync the repository, then preflight for Model v2
# @markdown Argument lists only, so no user value is ever interpolated into a shell string.

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/jxylxnn/knowing.git"  # @param {type:"string"}
# @markdown Model v2 is pushed on the V2-WIP branch; keep REPO_REF on that pushed ref.
REPO_REF = "V2-WIP"  # @param {type:"string"}
REPO_DIR = "knowing"  # @param {type:"string"}

# Files and CLI modules the rest of this notebook needs. If any are missing, this ref
# simply does not contain Model v2.
REQUIRED_V2_FILES = (
    "train.py",
    "canonicalize_data.py",
    "check_data.py",
    "simulate_season.py",
    "query_prob.py",
    "capture_official.py",
    "rebuild_snapshot.py",
    "config/model_v2.yaml",
    "src/training/v2_baseline.py",
    "src/models/bundle.py",
    "src/models/versioning.py",
    "src/simulation/v2_runner.py",
    "src/query/v2_probability.py",
    "src/data/snapshots.py",
    "src/data/nba_capture.py",
    "src/data/rebuild_snapshot.py",
    "src/features/snapshot_inputs.py",
    "src/forecasting/baseline_backend.py",
)

def run_command(command, cwd=None, check=True):
    '''Run one command from an argument list, echoing it and its output.'''
    print("$", " ".join(command))
    completed = subprocess.run(
        command, cwd=cwd, text=True, capture_output=True, check=False
    )
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    if check and completed.returncode:
        raise RuntimeError(
            f"Command failed with code {completed.returncode}: {' '.join(command)}"
        )
    return completed


REPO_PATH = os.path.abspath(REPO_DIR)
if not os.path.isdir(os.path.join(REPO_PATH, ".git")):
    run_command([
        "git", "clone", "--branch", REPO_REF, "--single-branch", REPO_URL, REPO_PATH,
    ])
else:
    run_command(["git", "fetch", "origin", REPO_REF], cwd=REPO_PATH)
    run_command(["git", "checkout", REPO_REF], cwd=REPO_PATH)
    run_command(["git", "merge", "--ff-only", f"origin/{REPO_REF}"], cwd=REPO_PATH)

COMMIT = run_command(["git", "rev-parse", "HEAD"], cwd=REPO_PATH).stdout.strip()
print(f"\nRepository ready at {REPO_PATH}")
print("REPO_REF :", REPO_REF)
print("COMMIT   :", COMMIT)

# --- Preflight 1: the ref must actually contain the v2 stack -----------------
missing = [name for name in REQUIRED_V2_FILES if not (Path(REPO_PATH) / name).is_file()]
if missing:
    raise RuntimeError(
        "This ref does not contain the Model v2 stack.\n"
        f"  REPO_REF checked : {REPO_REF} (commit {COMMIT or 'unknown'})\n"
        f"  Missing files    : {', '.join(missing)}\n"
        "Set REPO_REF (the first field of this cell) to the pushed V2-WIP ref that "
        "contains Model v2, then re-run this cell."
    )
print(f"\nPreflight: all {len(REQUIRED_V2_FILES)} required v2 files are present.")
print(
    "The CLI import preflight runs at the end of the dependency-install cell, because "
    "the v2 modules import pandas/nba_api/PyYAML at module level."
)

In [ ]:
# @title Mount Google Drive
# @markdown Drive keeps snapshots, bundles, and forecast exports across Colab sessions.
# @markdown With this off, the configuration cell falls back to a local root instead.

from pathlib import Path

MOUNT_DRIVE = True  # @param {type:"boolean"}

if MOUNT_DRIVE:
    from google.colab import drive  # type: ignore

    drive.mount("/content/drive")
else:
    print("Drive mount skipped; the configuration cell will use a local fallback root.")

print("Drive root exists:", Path("/content/drive/MyDrive").is_dir())

In [ ]:
# @title Install the lightweight v2 dependency set
# @markdown Only what the v2 baseline, query, and NBA capture paths actually load.
# @markdown Legacy extras (torch, catboost, lightgbm, efficient-kan) are not needed here.

import sys

if "run_command" not in globals():
    raise RuntimeError("Run the clone/sync cell first.")

INSTALL_DEPENDENCIES = True  # @param {type:"boolean"}
# Versions match the repo's pinned requirements for the v2 path.
V2_PACKAGES = "numpy==2.3.5 pandas==2.3.3 pyarrow==24.0.0 scipy==1.16.3 scikit-learn==1.8.0 PyYAML==6.0.3 joblib==1.5.3 psutil==7.2.2 nba_api==1.11.3 requests==2.32.5"  # @param {type:"string"}

# The v2 CLI modules import pandas/nba_api/PyYAML at module level, so the import
# preflight lives here rather than in the clone cell: before installation a fresh
# Colab cannot import them, and the check would fail for the wrong reason.
REQUIRED_V2_MODULES = (
    "train",
    "canonicalize_data",
    "check_data",
    "simulate_season",
    "query_prob",
    "capture_official",
    "rebuild_snapshot",
)

if INSTALL_DEPENDENCIES:
    packages = [name for name in V2_PACKAGES.split() if name]
    if not packages:
        raise RuntimeError("V2_PACKAGES is empty; nothing to install.")
    run_command(
        [sys.executable, "-m", "pip", "install", *packages], cwd=REPO_PATH
    )
else:
    print("Install skipped.")

import pandas as pd  # noqa: E402  (confirms the runtime is usable)

print("pandas", pd.__version__)

# --- Preflight: the v2 CLI modules must import cleanly now that deps exist ----
preflight_code = "\n".join([
    "import importlib",
    "for name in " + repr(REQUIRED_V2_MODULES) + ":",
    "    importlib.import_module(name)",
    "print('v2 CLI modules import cleanly')",
])
completed = run_command(
    [sys.executable, "-c", preflight_code], cwd=REPO_PATH, check=False
)
if completed.returncode:
    raise RuntimeError(
        "The v2 CLI modules are present but still do not import after installing "
        "V2_PACKAGES.\n"
        "Check the pip output above for a failed or conflicted package and re-run this "
        "cell with INSTALL_DEPENDENCIES = True. If the pip install never ran because this "
        "cell was skipped, run it now. If the modules are missing entirely, the ref does "
        "not contain Model v2: set REPO_REF to the pushed V2-WIP ref that does."
    )

In [ ]:
# @title Configure v2 paths and run parameters
# @markdown All paths are explicit; nothing is inferred from the working directory.

import os

if "run_command" not in globals():
    raise RuntimeError("Run the clone/sync cell first.")

DRIVE_ROOT = "/content/drive/MyDrive/nba_model"  # @param {type:"string"}
LOCAL_FALLBACK_ROOT = "/content/nba_model"  # @param {type:"string"}
DATA_DIR = DRIVE_ROOT + "/data"  # @param {type:"string"}
MODELS_DIR = DRIVE_ROOT + "/models"  # @param {type:"string"}
OUTPUT_DIR = DRIVE_ROOT + "/sim_results/v2"  # @param {type:"string"}
CONFIG_PATH = "config/model_v2.yaml"  # @param {type:"string"}

# @markdown ---
# @markdown **Snapshot handling:** leave `SNAPSHOT_ID` empty to create a fresh snapshot.
SNAPSHOT_ID = ""  # @param {type:"string"}
REUSE_EXISTING_SNAPSHOT = False  # @param {type:"boolean"}

# @markdown **Candidate identity:** `RUN_ID` becomes the immutable directory name.
RUN_ID = ""  # @param {type:"string"}

# @markdown ---
# @markdown **Simulation controls.** Degraded is the default so no 30-request roster capture happens.
HORIZON = "morning"  # @param ["previous_night", "morning", "pregame_90m", "pregame_30m"]
MODE = "degraded"  # @param ["official", "shadow", "degraded"]
SIM_WINDOW = "today"  # @param ["today", "date", "week", "season"]
SIM_DATE = ""  # @param {type:"string"}
SIMULATIONS = 100  # @param {type:"integer"}
SEED = 42  # @param {type:"integer"}

# @markdown ---
# @markdown **Official capture controls** (see the capture cell).
CAPTURE_SEASON = "2026-27"  # @param {type:"string"}
CAPTURE_OFFICIAL_ROSTERS = True  # @param {type:"boolean"}
ROSTER_TEAM_IDS = ""  # @param {type:"string"}
SKIP_SCHEDULE_CAPTURE_IF_COVERED = False  # @param {type:"boolean"}

# If Drive is not mounted, a /content/drive path would be a dead end; use the local root.
if not globals().get("MOUNT_DRIVE", False) and DRIVE_ROOT.startswith("/content/drive"):
    print(
        f"Drive is not mounted, so DRIVE_ROOT is falling back to {LOCAL_FALLBACK_ROOT}.\n"
        "Artifacts there are session-local and will not survive the Colab runtime."
    )
    DRIVE_ROOT = LOCAL_FALLBACK_ROOT
    DATA_DIR = DRIVE_ROOT + "/data"
    MODELS_DIR = DRIVE_ROOT + "/models"
    OUTPUT_DIR = DRIVE_ROOT + "/sim_results/v2"

for directory in (DATA_DIR, MODELS_DIR, OUTPUT_DIR):
    os.makedirs(directory, exist_ok=True)

print("Repository :", REPO_PATH)
print("Data dir   :", DATA_DIR)
print("Models dir :", MODELS_DIR)
print("Output dir :", OUTPUT_DIR)
print("Config     :", CONFIG_PATH)
print("Snapshot   :", SNAPSHOT_ID or "(created by the data cell)")
print("Mode       :", MODE)

In [ ]:
# @title Update source data and create (or reuse) a snapshot

import sys
from pathlib import Path

if "SNAPSHOT_ID" not in globals():
    raise RuntimeError("Run the configuration cell first.")


def newest_snapshot_id(data_dir):
    '''Return the most recently created source snapshot id, if any.'''
    manifests = sorted(
        Path(data_dir).glob("manifests/source_snapshot_*.json"),
        key=lambda path: path.stat().st_mtime,
    )
    if not manifests:
        return None
    return manifests[-1].stem[len("source_snapshot_"):]


if SNAPSHOT_ID and REUSE_EXISTING_SNAPSHOT:
    manifest = Path(DATA_DIR) / "manifests" / f"source_snapshot_{SNAPSHOT_ID}.json"
    if not manifest.is_file():
        raise FileNotFoundError(
            f"Requested snapshot does not exist: {SNAPSHOT_ID} ({manifest}). "
            "Clear SNAPSHOT_ID to create a fresh one."
        )
    print(f"Reusing existing snapshot {SNAPSHOT_ID}")
else:
    run_command(
        [
            sys.executable, "update_data.py",
            "--update",
            "--snapshot",
            "--data-dir", DATA_DIR,
        ],
        cwd=REPO_PATH,
    )
    SNAPSHOT_ID = newest_snapshot_id(DATA_DIR)
    if not SNAPSHOT_ID:
        raise RuntimeError(
            "No source snapshot was produced. Check the update_data.py output above."
        )
    print(f"\nCreated source snapshot: {SNAPSHOT_ID}")

print("SNAPSHOT_ID =", SNAPSHOT_ID)

In [ ]:
# @title Capture official tables and rebuild the snapshot
# @markdown Fresh schedule capture by default. Rosters only for official/shadow, in one rebuild.

import json
from datetime import date, timedelta
from pathlib import Path

if "SNAPSHOT_ID" not in globals():
    raise RuntimeError("Run the configuration cell first.")


def last_line(text):
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return lines[-1] if lines else ""


def target_dates(window, sim_date):
    '''The game dates the simulation window will ask for.'''
    if window == "date":
        if not sim_date.strip():
            raise RuntimeError("SIM_DATE must be set when SIM_WINDOW is 'date'.")
        start = date.fromisoformat(sim_date.strip())
    else:
        start = date.today()
    days = 7 if window == "week" else 30 if window == "season" else 1
    return [(start + timedelta(days=offset)).isoformat() for offset in range(days)]


def snapshot_schedule_dates(data_dir, snapshot_id):
    '''Dates covered by the snapshot schedule, or None when there is no schedule.'''
    inspect_code = "\n".join([
        "import json",
        "import sys",
        "import pandas as pd",
        "from src.features.snapshot_inputs import load_snapshot_schedule",
        "frame = load_snapshot_schedule(sys.argv[1], sys.argv[2])",
        "dates = sorted(set(pd.to_datetime(frame['GAME_DATE']).dt.date.astype(str)))",
        "print(json.dumps(dates))",
    ])
    completed = run_command(
        [sys.executable, "-c", inspect_code, data_dir, snapshot_id],
        cwd=REPO_PATH,
        check=False,
    )
    if completed.returncode:
        return None
    return json.loads(last_line(completed.stdout))


TARGET_DATES = target_dates(SIM_WINDOW, SIM_DATE)
print("Target game dates:", ", ".join(TARGET_DATES))

# Schedule capture: fresh by default; skipping requires proof of date coverage.
need_schedule = True
if SKIP_SCHEDULE_CAPTURE_IF_COVERED:
    covered = snapshot_schedule_dates(DATA_DIR, SNAPSHOT_ID)
    if covered is not None and set(TARGET_DATES) <= set(covered):
        need_schedule = False
        print(
            f"Snapshot {SNAPSHOT_ID} already covers every target date; skipping schedule "
            "capture (SKIP_SCHEDULE_CAPTURE_IF_COVERED is on)."
        )
    else:
        found = sorted(set(covered or []))
        print(
            "Coverage check failed, capturing a fresh schedule.\n"
            f"  target dates : {', '.join(TARGET_DATES)}\n"
            f"  snapshot has : {', '.join(found) if found else '(no usable schedule)'}"
        )
else:
    print("Capturing a fresh official schedule (default).")

# Roster capture: only the strict modes use official roster membership.
want_rosters = bool(CAPTURE_OFFICIAL_ROSTERS) and MODE in ("official", "shadow")
if MODE == "degraded":
    print(
        "Roster capture skipped: MODE=degraded uses appearance-derived rosters, and a full "
        "capture costs one rate-limited request per team."
    )
elif not CAPTURE_OFFICIAL_ROSTERS:
    print(f"Roster capture disabled by CAPTURE_OFFICIAL_ROSTERS for MODE={MODE}.")

captures = []

if need_schedule:
    captured = run_command(
        [
            sys.executable, "capture_official.py",
            "--native", "schedule",
            "--season", CAPTURE_SEASON,
            "--data-dir", DATA_DIR,
        ],
        cwd=REPO_PATH,
    )
    captures.append(last_line(captured.stdout))
    print(f"Schedule capture: {captures[-1]}")

if want_rosters:
    team_ids = [item.strip() for item in ROSTER_TEAM_IDS.split(",") if item.strip()]
    for team_id in team_ids:
        if not team_id.isdigit():
            raise RuntimeError(f"ROSTER_TEAM_IDS must be numeric team ids; got {team_id!r}")
    if not team_ids:
        print(
            "Capturing rosters for every team: one rate-limited request per team "
            "(roughly 30 requests, spaced about 1.2s apart)."
        )
    else:
        print(f"Capturing rosters for {len(team_ids)} team(s): {', '.join(team_ids)}")
    roster_command = [
        sys.executable, "capture_official.py",
        "--native", "rosters",
        "--season", CAPTURE_SEASON,
        "--data-dir", DATA_DIR,
    ]
    for team_id in team_ids:
        roster_command += ["--team-id", team_id]
    captured = run_command(roster_command, cwd=REPO_PATH)
    captures.append(last_line(captured.stdout))
    print(f"Roster capture: {captures[-1]}")
    if len(set(TARGET_DATES)) > 1:
        print(
            "Warning: this capture only certifies requests whose forecast cutoff falls on "
            "the roster capture's own Eastern date, so a strict "
            f"{SIM_WINDOW} window covering {len(set(TARGET_DATES))} local dates needs a "
            "separate point-in-time snapshot and roster capture per date. Strict single-day "
            "runs are the supported shape; use MODE=degraded for multi-day windows."
        )

if captures:
    rebuild_command = [
        sys.executable, "rebuild_snapshot.py",
        "--base-snapshot-id", SNAPSHOT_ID,
        "--data-dir", DATA_DIR,
    ]
    for capture in captures:
        rebuild_command += ["--capture", capture]
    rebuilt = run_command(rebuild_command, cwd=REPO_PATH)
    rebuilt_id = last_line(rebuilt.stdout)
    if not rebuilt_id:
        raise RuntimeError("rebuild_snapshot.py did not report a new snapshot id.")
    print(f"Base snapshot    : {SNAPSHOT_ID}")
    print(f"Rebuilt snapshot : {rebuilt_id}")
    SNAPSHOT_ID = rebuilt_id
else:
    print("No captures taken; the snapshot is unchanged.")

manifest_path = Path(DATA_DIR) / "manifests" / f"source_snapshot_{SNAPSHOT_ID}.json"
if manifest_path.is_file():
    created_at = json.loads(manifest_path.read_text(encoding="utf-8")).get("created_at")
    print("\nSnapshot created_at:", created_at)

print(
    "\nHorizon cutoff rule: a snapshot is usable for a horizon only when its manifest "
    "created_at AND every file's receipt time are at or before that horizon's cutoff "
    "(previous_night = 23:59 ET the day before; morning = 09:00 ET on the game date). "
    "A rebuild stamps created_at at rebuild time, so finish the rebuild before the cutoff "
    "you intend to forecast at. The simulation cell preflights this and refuses to launch "
    "a run that cannot satisfy it."
)
print("SNAPSHOT_ID =", SNAPSHOT_ID)

In [ ]:
# @title Canonicalize and check the snapshot

import sys

if "SNAPSHOT_ID" not in globals():
    raise RuntimeError("Run the configuration cell first.")

run_command(
    [
        sys.executable, "canonicalize_data.py",
        "--snapshot-id", SNAPSHOT_ID,
        "--data-dir", DATA_DIR,
    ],
    cwd=REPO_PATH,
)

run_command(
    [
        sys.executable, "check_data.py",
        "--snapshot-id", SNAPSHOT_ID,
        "--data-dir", DATA_DIR,
    ],
    cwd=REPO_PATH,
)

print(f"\nSnapshot {SNAPSHOT_ID} is canonicalized and checked.")

In [ ]:
# @title Train and validate an immutable v2 candidate bundle
# @markdown Only current v2 arguments; the baseline is never promotion eligible.

import json
import sys
from pathlib import Path

if "SNAPSHOT_ID" not in globals():
    raise RuntimeError("Run the configuration cell first.")

CHECK_BACKEND_LOAD = True  # @param {type:"boolean"}

train_command = [
    sys.executable, "train.py",
    "--architecture", "v2",
    "--preset", "baseline",
    "--snapshot-id", SNAPSHOT_ID,
    "--data-dir", DATA_DIR,
    "--models-dir", MODELS_DIR,
    "--config", CONFIG_PATH,
    "--json",
]
if RUN_ID.strip():
    train_command += ["--run-id", RUN_ID.strip()]

completed = run_command(train_command, cwd=REPO_PATH, check=False)
if completed.returncode:
    raise RuntimeError(
        f"train.py failed with code {completed.returncode}; see its output above."
    )

payload = None
try:
    payload = json.loads(completed.stdout[completed.stdout.index("{"):])
except (ValueError, json.JSONDecodeError):
    payload = None

if payload:
    print("Training status :", payload.get("status"))
    print("Promoted        :", payload.get("promoted"))


def candidate_from_run_id(models_dir, run_id):
    if not run_id:
        return None
    path = Path(models_dir) / "versions" / run_id
    return path if path.is_dir() else None


def newest_candidate(models_dir):
    versions = Path(models_dir) / "versions"
    if not versions.is_dir():
        return None
    sealed = [
        path for path in versions.iterdir()
        if (path / "bundle_manifest.json").is_file()
    ]
    if not sealed:
        return None
    return max(sealed, key=lambda path: path.stat().st_mtime)


# The success payload may or may not carry the candidate path, so fall back to the
# explicit run id and then to the newest sealed bundle under models/versions.
CANDIDATE_DIR = None
if payload and payload.get("candidate"):
    reported = Path(str(payload["candidate"]))
    if (reported / "bundle_manifest.json").is_file():
        CANDIDATE_DIR = reported
if CANDIDATE_DIR is None:
    CANDIDATE_DIR = candidate_from_run_id(MODELS_DIR, RUN_ID.strip())
if CANDIDATE_DIR is None:
    CANDIDATE_DIR = newest_candidate(MODELS_DIR)

if CANDIDATE_DIR is None:
    raise RuntimeError(
        "Training reported success but no sealed candidate bundle was found under "
        f"{Path(MODELS_DIR) / 'versions'}"
    )

CANDIDATE_DIR = str(Path(CANDIDATE_DIR).resolve())
print("\nCANDIDATE_DIR =", CANDIDATE_DIR)

# Integrity check: the real v2 validator, with the promotion gate left alone.
validation_code = "\n".join([
    "import json",
    "import sys",
    "from src.models.bundle import validate_v2_bundle",
    "report = validate_v2_bundle(sys.argv[1], promotion=False)",
    "print(json.dumps(report.to_dict(), indent=2, sort_keys=True))",
])
if CHECK_BACKEND_LOAD:
    validation_code += "\n" + "\n".join([
        "from src.forecasting.baseline_backend import V2BaselineBackend",
        "V2BaselineBackend(sys.argv[1])",
        "print('sealed bundle loads in a clean process')",
    ])

run_command([sys.executable, "-c", validation_code, CANDIDATE_DIR], cwd=REPO_PATH)

print(
    "\nNote: `promote_model.py --dry-run` reports ineligible for this baseline by design "
    "(no point-in-time roster/status data, no multi-fold replay evidence). That is the "
    "promotion gate working, not an integrity failure: integrity is what the validator "
    "just measured. Promotion stays out of scope until those official inputs exist."
)

In [ ]:
# @title Simulate scheduled Model v2 games
# @markdown Preflights the horizon cutoff first; degraded needs both candidate and flag.

import json
import sys
from datetime import date, timedelta

if "SNAPSHOT_ID" not in globals():
    raise RuntimeError("Run the configuration cell first.")


def preflight_dates(window, sim_date):
    if window == "date":
        if not sim_date.strip():
            raise RuntimeError("SIM_DATE must be set when SIM_WINDOW is 'date'.")
        start = date.fromisoformat(sim_date.strip())
    else:
        start = date.today()
    days = 7 if window == "week" else 30 if window == "season" else 1
    return start.isoformat(), days


START_DATE, DAYS = preflight_dates(SIM_WINDOW, SIM_DATE)

# Mirrors SourceSnapshotManifest.assert_available_at: both the manifest created_at and
# every file receipt time must be at or before the horizon cutoff for a game.
preflight_code = "\n".join([
    "import json",
    "import sys",
    "from datetime import timedelta",
    "import pandas as pd",
    "from src.data.snapshots import load_source_snapshot_manifest",
    "from src.features.snapshot_inputs import load_snapshot_schedule",
    "from src.simulation.v2_runner import horizon_cutoff",
    "data_dir = sys.argv[1]",
    "snapshot_id = sys.argv[2]",
    "horizon = sys.argv[3]",
    "start_date = pd.Timestamp(sys.argv[4]).date()",
    "days = int(sys.argv[5])",
    "targets = {(start_date + timedelta(days=offset)).isoformat() for offset in range(days)}",
    "manifest = load_source_snapshot_manifest(data_dir, snapshot_id)",
    "created = manifest.created_datetime",
    "file_times = [",
    "    pd.Timestamp(record.fetched_at or manifest.created_at).to_pydatetime()",
    "    for record in manifest.files",
    "]",
    "frame = load_snapshot_schedule(data_dir, snapshot_id)",
    "frame = frame.assign(GAME_DATE=pd.to_datetime(frame['GAME_DATE']).dt.date.astype(str))",
    "frame = frame.loc[frame['GAME_DATE'].isin(targets)]",
    "cutoffs = [",
    "    horizon_cutoff(pd.Timestamp(value).to_pydatetime(), horizon)",
    "    for value in frame['SCHEDULED_TIP']",
    "]",
    "forecastable = 0",
    "for cutoff in cutoffs:",
    "    if created <= cutoff and all(value <= cutoff for value in file_times):",
    "        forecastable += 1",
    "print(json.dumps({",
    "    'games': int(len(frame)),",
    "    'forecastable': int(forecastable),",
    "    'created_at': created.isoformat(),",
    "    'earliest_cutoff': min(cutoffs).isoformat() if cutoffs else None,",
    "}))",
])

completed = run_command(
    [
        sys.executable, "-c", preflight_code,
        DATA_DIR, SNAPSHOT_ID, HORIZON, START_DATE, str(DAYS),
    ],
    cwd=REPO_PATH,
    check=False,
)
if completed.returncode:
    raise RuntimeError(
        f"Snapshot {SNAPSHOT_ID} has no usable schedule for {START_DATE} "
        f"(+{DAYS - 1} day(s)). Run the capture cell to archive an official schedule and "
        "rebuild the snapshot before simulating."
    )

lines = [line.strip() for line in completed.stdout.splitlines() if line.strip()]
info = json.loads(lines[-1])
print("Preflight (horizon %s):" % HORIZON)
print("  games in window   :", info["games"])
print("  forecastable      :", info["forecastable"])
print("  snapshot created  :", info["created_at"])
print("  earliest cutoff   :", info["earliest_cutoff"])

if info["games"] == 0:
    raise RuntimeError(
        f"The snapshot schedule has no games on {START_DATE} (+{DAYS - 1} day(s)). "
        "Capture a schedule for the right season or change SIM_WINDOW/SIM_DATE."
    )
if info["forecastable"] == 0:
    raise RuntimeError(
        "No game in this window can be forecast at horizon "
        f"{HORIZON!r}: the snapshot (created {info['created_at']}) is newer than every "
        f"cutoff (earliest {info['earliest_cutoff']}).\n"
        "A snapshot is only usable when it was created and received before the horizon "
        "cutoff: previous_night = 23:59 ET the day before, morning = 09:00 ET on the game "
        "date. Capture and rebuild before that cutoff, choose an earlier horizon, or "
        "simulate a later date."
    )
if info["forecastable"] < info["games"]:
    print(
        f"\nWarning: only {info['forecastable']} of {info['games']} games satisfy the "
        "cutoff, so the run may fail partway. Prefer a window whose games are all after "
        "the snapshot time."
    )

sim_command = [sys.executable, "simulate_season.py"]

if SIM_WINDOW == "today":
    sim_command += ["--today"]
elif SIM_WINDOW == "date":
    sim_command += ["--date", SIM_DATE.strip()]
elif SIM_WINDOW == "week":
    sim_command += ["--week"]
else:
    sim_command += ["--season"]

sim_command += [
    "--snapshot-id", SNAPSHOT_ID,
    "--horizon", HORIZON,
    "--data-dir", DATA_DIR,
    "--models-dir", MODELS_DIR,
    "--output-dir", OUTPUT_DIR,
    "--sims", str(SIMULATIONS),
    "--seed", str(SEED),
    "--json",
]

candidate_path = globals().get("CANDIDATE_DIR")
if MODE in ("shadow", "degraded"):
    if not candidate_path:
        raise RuntimeError(
            f"MODE={MODE} needs a sealed candidate; run the training cell first."
        )
    sim_command += ["--candidate", candidate_path]
if MODE == "degraded":
    sim_command += ["--allow-degraded"]

if MODE == "official":
    print(
        "\nRequested mode : official (requires point-in-time roster membership; publishes "
        "to the official ledger)"
    )
elif MODE == "shadow":
    print("\nRequested mode : shadow (sealed candidate; never publishes)")
else:
    print("\nRequested mode : degraded (appearance-derived rosters; never publishes)")

completed = run_command(sim_command, cwd=REPO_PATH, check=False)
if completed.returncode:
    raise RuntimeError(
        f"simulate_season.py failed with code {completed.returncode}; "
        "see its output above."
    )

payload = None
try:
    payload = json.loads(completed.stdout[completed.stdout.index("{"):])
except (ValueError, json.JSONDecodeError):
    payload = None

FORECAST_PATH = None
SAMPLES_PATH = None
if payload:
    print("\nReported mode      :", payload.get("mode"))
    print("Official           :", payload.get("official"))
    print(
        "Ledger flag (advisory, mode classification only):",
        payload.get("published_to_official_ledger"),
    )
    print("Bundle             :", payload.get("model_bundle_id"))
    print("Games              :", payload.get("games"))
    FORECAST_PATH = payload.get("forecast_path")
    SAMPLES_PATH = payload.get("samples_path")
    print("Forecasts          :", FORECAST_PATH)
    print("Samples            :", SAMPLES_PATH)

In [ ]:
# @title Locate the immutable forecast and sample files
# @markdown Exports are content-addressed, so each run lands in its own directory.

from pathlib import Path

if "OUTPUT_DIR" not in globals():
    raise RuntimeError("Run the configuration cell first.")

# A complete export has both files; a half-written directory is not a usable run.
RUN_DIRS = sorted(
    (
        path
        for path in Path(OUTPUT_DIR).glob("*")
        if (path / "forecasts.parquet").is_file()
        and (path / "samples.parquet").is_file()
    ),
    key=lambda path: path.stat().st_mtime,
)
if not RUN_DIRS:
    raise RuntimeError(
        f"No complete forecast export (forecasts.parquet and samples.parquet) found under "
        f"{OUTPUT_DIR}. Run the simulation cell first."
    )

LATEST_RUN_DIR = RUN_DIRS[-1]
if not globals().get("FORECAST_PATH"):
    FORECAST_PATH = str(LATEST_RUN_DIR / "forecasts.parquet")
if not globals().get("SAMPLES_PATH"):
    SAMPLES_PATH = str(LATEST_RUN_DIR / "samples.parquet")

print("Forecast export dir:", LATEST_RUN_DIR)
print("Complete runs found:", len(RUN_DIRS))
print("FORECAST_PATH      :", FORECAST_PATH)
print("SAMPLES_PATH       :", SAMPLES_PATH)

In [ ]:
# @title Query a player probability
# @markdown Edit the three fields below and run the cell.

PLAYER = "LeBron James"  # @param {type:"string"}
STAT = "pts"  # @param ["pts", "reb", "ast", "stl", "blk", "tov"]
LINE = 25.5  # @param {type:"number"}

import sys

if "DATA_DIR" not in globals():
    raise RuntimeError("Run the configuration cell first.")

if not globals().get("FORECAST_PATH"):
    raise RuntimeError("No forecast file yet; run the simulation cell, then locate it.")

query_command = [
    sys.executable, "query_prob.py",
    "--player", PLAYER,
    "--stat", STAT,
    "--line", str(LINE),
    "--forecast-file", FORECAST_PATH,
    "--players-file", f"{DATA_DIR}/nba_players.csv",
    "--json",
]

completed = run_command(query_command, cwd=REPO_PATH, check=False)
if completed.returncode:
    print(f"\nquery_prob.py exited with code {completed.returncode} (see output above).")

In [ ]:
# @title Optional: bundle the candidate and export artifacts

import shutil
from pathlib import Path

if "DRIVE_ROOT" not in globals():
    raise RuntimeError("Run the configuration cell first.")

BUNDLE_CANDIDATE = True  # @param {type:"boolean"}
BACKUP_FORECASTS = True  # @param {type:"boolean"}

EXPORT_DIR = Path(DRIVE_ROOT) / "exports"

if BUNDLE_CANDIDATE:
    if not globals().get("CANDIDATE_DIR"):
        raise RuntimeError("No candidate to bundle; run the training cell first.")
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    archive_base = EXPORT_DIR / Path(CANDIDATE_DIR).name
    print("Candidate archive:", shutil.make_archive(str(archive_base), "zip", CANDIDATE_DIR))

if BACKUP_FORECASTS:
    if not globals().get("LATEST_RUN_DIR"):
        raise RuntimeError("No forecast export to back up; run the locate cell first.")
    backup_dir = EXPORT_DIR / Path(LATEST_RUN_DIR).name
    backup_dir.mkdir(parents=True, exist_ok=True)
    for name in ("forecasts.parquet", "samples.parquet", "manifest.json"):
        source = Path(LATEST_RUN_DIR) / name
        if source.is_file():
            shutil.copy2(source, backup_dir / name)
    print("Forecast backup:", backup_dir)

print("\nArtifacts stay on Drive; this cell deletes nothing.")